In [1]:
!pip install transformers bitsandbytes peft accelerate datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.8 MB/s eta 0:00:00


In [2]:
from datasets import Dataset
import shutil

In [3]:
import json

with open("/content/dpo_dataset.json", "r") as f:
  data = json.load(f)

combined_data = []
for entry in data:
  combined_entry = f"Prompt: {entry["prompt"]}.........Chosen: {entry["chosen"]}.........Rejected: {entry["rejected"]}"
  combined_data.append(combined_entry)



In [4]:
data = [{'text': entry} for entry in combined_data]

In [5]:
dataset = Dataset.from_list(data)

In [6]:
dataset

Dataset({
    features: ['text'],
    num_rows: 100
})

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

In [8]:
base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [9]:
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [10]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [11]:
def tokenize_fn(examples):
  tokens = tokenizer(examples['text'], truncation=True, padding='max_length', max_length=512)
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

In [12]:
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [13]:
tokenized

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 100
})

In [16]:
instruction_model_zip_path = "/content/instructional_model.zip"

In [17]:
shutil.unpack_archive(instruction_model_zip_path, instruction_model_zip_path.split(".zip")[0])

In [18]:
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto"
    )

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [19]:
model = PeftModel.from_pretrained(base_model, instruction_model_zip_path.split(".zip")[0])
model.enable_input_require_grads()

In [20]:
training_args = TrainingArguments(
    output_dir="/content/trained_model",
    per_device_train_batch_size=2,
    num_train_epochs=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    report_to="none")

In [21]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized
)

In [22]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
50,5.155919
100,5.154390


TrainOutput(global_step=100, training_loss=5.155154113769531, metrics={'train_runtime': 170.9473, 'train_samples_per_second': 1.17, 'train_steps_per_second': 0.585, 'total_flos': 636296468889600.0, 'train_loss': 5.155154113769531, 'epoch': 2.0})

In [29]:
from google.colab import files

# shutil.make_archive(output_name, format, folder_to_zip)
shutil.make_archive('/content/DPO_model', 'zip', '/content/trained_model/checkpoint-100')

files.download('/content/DPO_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
prompt = "How to reach the non-target customers"

In [25]:
formatted_prompt = f"Question: {prompt}\nAnswer: "

In [26]:
input = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

In [30]:
outputs = model.generate(
    **input,
    max_new_tokens=200,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
    repetition_penalty=1.1
)

In [31]:
tokenizer.decode(outputs[0], skip_special_tokens=True)

"Question: How to reach the non-target customers\nAnswer: 1. Use customer segmentation: Segment your customers based on their behavior, demographics, purchase history, and other relevant information. Use this segmented data to create customized messages that are more likely to resonate with them.\n2. Create personalized emails: Use email marketing software like MailChimp or Constant Contact to create targeted and engaging emails that include relevant product recommendations and offers.\n3. Personalize the email subject line: The subject line of an email is critical to its success. Ensure it's attention-grabbing, related to the message and has the right tone.\n4. Use dynamic content: Dynamic content is a feature in email marketing software that allows you to dynamically change content according to the recipient's behavior. This can lead to better engagement rates.\n5. Test and optimize: Regularly test different campaigns and messages to see what works best for your target audience. Opti